# Week 4 Day 4 - Deep Agents, the top layer

We have reached the top of the stack. A Deep Agent takes the create_agent you met yesterday and wraps it in an opinionated harness, the kind of setup you would otherwise build yourself for serious, multi-step work.

Three things come built in. A planning tool, so the agent can write itself a todo list and work through it. A filesystem, so it can save and reread its working notes rather than holding everything in the conversation. And sub-agents, so it can hand a focused piece of work to a helper with its own clean context. You supply the intent, and the harness supplies the structure.

## When to reach for this

For a quick question with one or two tools, create_agent is the right tool and a Deep Agent would be overkill. Deep Agents earn their keep when a task has many steps, produces artifacts, and benefits from the agent organising its own work. Research and report writing is the classic example, so that is what we will build.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Before you run this</h2>
            <span style="color:#ff7800;">This lab needs <code>OPENAI_API_KEY</code> and a <code>SERPER_API_KEY</code> for web search. The agent writes its work into a <code>sandbox</code> folder next to this notebook, which we create for it below.
            </span>
        </td>
    </tr>
</table>

In [ ]:
# Imports and environment first

import os
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

load_dotenv(override=True)

## A web search tool

We give the agent one tool of our own, a web search backed by Serper. We call the API directly with `requests` so you can see exactly what it does.

In [ ]:
@tool
def search(query: str) -> str:
    """Search the web for current information and return the top results."""
    response = requests.post(
        "https://google.serper.dev/search",
        headers={"X-API-KEY": os.getenv("SERPER_API_KEY"), "Content-Type": "application/json"},
        json={"q": query},
    )
    data = response.json()
    parts = []
    if "answerBox" in data:
        box = data["answerBox"]
        parts.append(box.get("answer") or box.get("snippet") or "")
    for item in data.get("organic", [])[:5]:
        parts.append(f"{item.get('title')}: {item.get('snippet')}")
    return "\n".join(p for p in parts if p)

## A research agent that plans, searches and writes

We point the agent's filesystem at a real `sandbox` folder, so anything it writes appears as a file on disk that we can open afterwards. The agent sees its filesystem rooted at `/`, so when it reports writing to `/webb.md`, that lands as `webb.md` inside our sandbox folder, which is why we read it from there below. Then we give it a research brief.

Watch what it does. It will write itself a todo list, search the web a few times, and save a briefing to a file, all without us managing any of those steps.

In [ ]:
sandbox = os.path.abspath("sandbox")
os.makedirs(sandbox, exist_ok=True)

model = ChatOpenAI(model="gpt-5.4-mini")

researcher = create_deep_agent(
    model=model,
    tools=[search],
    system_prompt=(
        "You are a research assistant. Plan your work with your todo tool, "
        "research with the search tool, and write your findings as a tidy markdown briefing to a file."
    ),
    backend=FilesystemBackend(root_dir=sandbox, virtual_mode=True),
)

In [ ]:
brief = (
    "Research the James Webb Space Telescope. Find out when it launched and two of its major discoveries. "
    "Write a one page markdown briefing, with a heading and a short section for each, to the file webb.md."
)

result = researcher.invoke({"messages": [{"role": "user", "content": brief}]})
print(result["messages"][-1].content)

Let us see how it worked. First, the tools it chose to call, which reveal the planning and the file writing. Then the briefing itself, read straight back from the sandbox folder.

In [ ]:
tools_used = [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", []) or [])]
print("Tools the agent called, in order:")
print(tools_used)

In [ ]:
display(Markdown(open(os.path.join(sandbox, "webb.md")).read()))

## Sub-agents: handing off focused work

A sub-agent is a helper the main agent can delegate to through its `task` tool. The helper runs with its own fresh context, does one job, and reports back a tidy result. This keeps the main agent's attention clear when a task has several independent pieces.

We define a researcher sub-agent, then ask the main agent to compare two planets by delegating the research on each.

In [ ]:
research_subagent = {
    "name": "planet-researcher",
    "description": "Researches a single planet and returns a short list of facts about it.",
    "system_prompt": "You research one planet using the search tool and return three concise, interesting facts.",
}

lead = create_deep_agent(
    model=model,
    tools=[search],
    system_prompt=(
        "You write comparison briefings. For each subject, delegate the research to your planet-researcher "
        "sub-agent, then write a markdown comparison to a file."
    ),
    subagents=[research_subagent],
    backend=FilesystemBackend(root_dir=sandbox, virtual_mode=True),
)

result = lead.invoke({"messages": [{"role": "user", "content":
    "Compare Mars and Venus. Research each planet, then write a short markdown comparison to planets.md."}]})

tools_used = [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", []) or [])]
print("Tools the lead agent called:", tools_used)

In [ ]:
display(Markdown(open(os.path.join(sandbox, "planets.md")).read()))

## Recap, and where we are heading

You have run a Deep Agent that planned its own work, searched the web, wrote files to disk, and delegated to a sub-agent. That is a lot of capability for very little code, which is what the harness buys you.

Notice that higher is not always the right choice. Tomorrow's Sidekick deliberately drops back to the create_agent layer, because that is the right altitude for a responsive assistant. This is the stack-not-a-ladder idea in action: you pick the layer that fits the job.

Tomorrow we build the Sidekick, the project this whole week has been leading to. It mixes the layers freely: a create_agent worker, a browser and a filesystem through MCP, human-in-the-loop, and a UI you can talk to.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Give the research agent a second tool. If you add the headful browser tools from yesterday's MCP server, remember they are asynchronous, so you will switch <code>researcher.invoke(...)</code> to <code>await researcher.ainvoke(...)</code> as you did on Day 3. Then open the sandbox folder and read every file the agent wrote for itself along the way.
            </span>
        </td>
    </tr>
</table>